In [11]:
import pyvisa
rm = pyvisa.ResourceManager()
print(rm.list_resources())
vna=rm.open_resource('USB0::0x2A8D::0x2A01::MY58421887::0::INSTR')
print(vna.query('*IDN?'))

('ASRL5::INSTR', 'GPIB0::5::INSTR', 'GPIB1::1::INSTR', 'USB0::0x0957::0x0407::MY44045981::0::INSTR', 'USB0::0x2A8D::0x2A01::MY58421887::0::INSTR', 'ASRL1::INSTR', 'ASRL3::INSTR')
Keysight Technologies,N5222B,MY58421887,A.13.50.09



In [40]:


vna.clear()
vna.write('*CLS')

vna.write('SYST:FPR')
vna.write("CALC:PAR:DEF:EXT 'm1','S21'")
vna.write("CALC:PAR:SEL 'm1'")
vna.write('DISP:WIND1:STATE ON')

vna.write('SENS:FREQ:STAR 6e9'); vna.write('SENS:FREQ:STOP 7e9')
vna.write('SENS:SWE:POIN 201')
vna.write('SOUR:POW -30W')
vna.write('INIT:IMM'); vna.query('*OPC?') #trigger one sweep

vna.write

while True:
    err = vna.query('SYST:ERR?').strip()
    print(err)
    if err.startswith(('0,','+0')):
        break


-131,"Invalid suffix"
-213,"Init ignored"
+0,"No error"


In [1]:
import importlib, capture_vna_format
importlib.reload(capture_vna_format)
capture_vna_format.main([])

N5222B SCPI FORMAT CAPTURE  (read-only)
timestamp : 2026-07-20T11:16:06
address   : USB0::0x2A8D::0x2A01::MY58421887::0::INSTR

--- VISA resources visible on this PC ---
  ASRL5::INSTR
  GPIB0::5::INSTR
  GPIB1::1::INSTR
  USB0::0x0957::0x0407::MY44045981::0::INSTR
  USB0::0x2A8D::0x2A01::MY58421887::0::INSTR
  ASRL1::INSTR
  ASRL3::INSTR

--- SETTINGS / IDENTITY ---
IDN              | *IDN?                      -> 'Keysight Technologies,N5222B,MY58421887,A.13.50.09'
                 | ^ Confirms we are talking to the right box and gives the firmware rev.
DATA_FORMAT      | FORM:DATA?                 -> 'ASC,0'
                 | ^ ASC,0 = ASCII text. REAL,32 / REAL,64 = binary floats. Decides how we parse.
BYTE_ORDER       | FORM:BORD?                 -> 'NORM'
                 | ^ NORM = big-endian, SWAP = little-endian. Wrong choice = garbage numbers.
MEAS_CATALOG     | CALC:PAR:CAT:EXT?          -> '"CH1_S11_1,S21"'
                 | ^ Lists the measurement names and their S-param